# Taller Práctico — Auditoría de Código: Costo Computacional
### Profesor: Lucas Gómez Tobón

---

## Contexto

Acabas de ser contratado como analista cuantitativo en un fondo de inversión. Tu predecesor dejó un script de Python que genera el **reporte de riesgo diario**. El problema es que a medida que el fondo ha crecido y procesa más datos, el script se ha vuelto cada vez más lento. Hoy en día, **tarda horas en ejecutarse**, paralizando las decisiones del equipo de *trading*.

Tu misión es auditar el código heredado, identificar las ineficiencias, y reescribir cada función para que sea óptima.

El script original realiza tres tareas fundamentales:

1. **Limpieza de Datos:** Busca transacciones duplicadas en el registro diario.
2. **Análisis Técnico:** Calcula el promedio móvil de los precios de los activos.
3. **Reconciliación:** Cruza la cartera de clientes del fondo con una lista de clientes sancionados.

Al final, consolidarás todas tus funciones optimizadas en una función `generar_reporte_eficiente()` que reemplaza al código heredado.

## Instrucciones Generales

Para **cada problema** debes:

1. **Leer y entender** la función ineficiente que se te entrega. **No la modifiques ni la borres** — la usarás como referencia y para comparar resultados.
2. **Diagnosticar** la complejidad actual del algoritmo usando notación Big O. Identifica qué operaciones están causando la ineficiencia.
3. **Determinar** a qué complejidad objetivo se debe reducir el algoritmo.
4. **Implementar** una nueva función eficiente (con un nombre distinto) que produzca **exactamente los mismos resultados** que la original.
5. **Medir** el tiempo de ejecución de ambas funciones y calcular cuántas veces más rápida es tu versión.

> **Importante:** No destruyas el código viejo. Cada función ineficiente debe permanecer intacta como referencia. Tu trabajo es crear funciones **nuevas** que la reemplacen.

## Setup: Generación de Datos de Prueba

Ejecuta la siguiente celda para generar los datos simulados que usarás en todo el taller.

In [1]:
import time
import random

# Semilla para reproducibilidad
random.seed(42)

# --- Datos para Problema 1: Transacciones duplicadas ---
N_TXS = 15000
transacciones = list(range(1, N_TXS + 1))
# Inyectamos duplicados intencionales
transacciones.extend([1505, 8999, 12000, 3742, 9001])
random.shuffle(transacciones)

# --- Datos para Problema 2: Promedios móviles ---
N_PRECIOS = 20000
VENTANA = 1000
precios = [random.uniform(100, 200) for _ in range(N_PRECIOS)]

# --- Datos para Problema 3: Cruce de clientes ---
N_CLIENTES = 15000
clientes_cartera = [random.randint(1, N_CLIENTES * 2) for _ in range(N_CLIENTES)]
clientes_sancionados = [random.randint(1, N_CLIENTES * 2) for _ in range(N_CLIENTES)]

print(f"Datos generados:")
print(f"  - Transacciones: {len(transacciones):,} registros ({N_TXS:,} únicos + 5 duplicados)")
print(f"  - Precios históricos: {N_PRECIOS:,} días")
print(f"  - Ventana del promedio móvil: {VENTANA} días")
print(f"  - Clientes en cartera: {N_CLIENTES:,}")
print(f"  - Clientes sancionados: {N_CLIENTES:,}")

Datos generados:
  - Transacciones: 15,005 registros (15,000 únicos + 5 duplicados)
  - Precios históricos: 20,000 días
  - Ventana del promedio móvil: 1000 días
  - Clientes en cartera: 15,000
  - Clientes sancionados: 15,000


---

## Problema 1: Búsqueda de Transacciones Duplicadas

El sistema de registro del fondo a veces genera entradas duplicadas por errores de red. Tu primera tarea es identificar qué transacciones están duplicadas.

### Código heredado (ineficiente)

Analiza la siguiente función. Identifica:
- ¿Cuál es la complejidad actual en notación Big O?
- ¿Por qué es ineficiente?
- ¿A qué complejidad se puede reducir y con qué estructura de datos?

In [2]:
# ============================================================
# CÓDIGO HEREDADO — NO MODIFICAR
# ============================================================

def buscar_duplicados_fuerza_bruta(lista_tx):
    """
    Busca transacciones duplicadas comparando cada elemento con todos los demás.
    
    Parámetros:
    lista_tx (list): Lista de IDs de transacciones.
    
    Retorna:
    list: Lista con los IDs duplicados encontrados.
    """
    duplicados = []
    n = len(lista_tx)
    
    for i in range(n):
        for j in range(i + 1, n):
            if lista_tx[i] == lista_tx[j]:
                if lista_tx[i] not in duplicados:
                    duplicados.append(lista_tx[i])
                    
    return duplicados

### Tu tarea

Crea una función `buscar_duplicados_eficiente(lista_tx)` que devuelva exactamente los mismos resultados pero con una complejidad significativamente menor. Piensa: ¿qué estructura de datos te permite saber si ya viste un elemento en tiempo $O(1)$?

In [3]:
# ============================================================
# TU SOLUCIÓN
# ============================================================

def buscar_duplicados_eficiente(lista_tx):
    """
    Busca transacciones duplicadas utilizando un diccionario como registro.
    
    Parámetros:
    lista_tx (list): Lista de IDs de transacciones.
    
    Retorna:
    list: Lista con los IDs duplicados encontrados.
    """
    vistos = {}      # Diccionario para registrar lo que ya vimos
    duplicados = []
    
    for tx in lista_tx:
        if tx in vistos:
            if tx not in duplicados:
                duplicados.append(tx)
        else:
            vistos[tx] = True
    
    return duplicados

### Solución — Análisis de Complejidad

| | Código Heredado | Código Optimizado |
| :--- | :--- | :--- |
| **Complejidad** | $O(n^2)$ | $O(n)$ |
| **Razón** | Dos loops anidados: para cada transacción, recorre todas las demás. Además, `if tx not in duplicados` sobre una lista es $O(d)$ donde $d$ es el número de duplicados encontrados. | Un solo recorrido de la lista. Usa un diccionario donde la búsqueda (`in`) es $O(1)$. |
| **Estrategia** | Fuerza bruta | **Memoria** — Usar un diccionario como "libreta de registro" para recordar lo que ya vimos. |

In [4]:
# ============================================================
# COMPARACIÓN DE RESULTADOS
# ============================================================

print("--- Problema 1: Búsqueda de Duplicados ---\n")

t0 = time.time()
res_heredado = buscar_duplicados_fuerza_bruta(transacciones)
t1 = time.time()
tiempo_heredado = t1 - t0
print(f"Código heredado:   {tiempo_heredado:.4f} segundos | Duplicados: {sorted(res_heredado)}")

t0 = time.time()
res_eficiente = buscar_duplicados_eficiente(transacciones)
t1 = time.time()
tiempo_eficiente = t1 - t0
print(f"Tu solución:       {tiempo_eficiente:.6f} segundos | Duplicados: {sorted(res_eficiente)}")

# Verificación
assert sorted(res_heredado) == sorted(res_eficiente), "ERROR: Los resultados no coinciden."

speedup_1 = tiempo_heredado / tiempo_eficiente if tiempo_eficiente > 0 else float('inf')
print(f"\nGanancia en eficiencia: {speedup_1:.0f}x más rápido")

--- Problema 1: Búsqueda de Duplicados ---

Código heredado:   1.9732 segundos | Duplicados: [1505, 3742, 8999, 9001, 12000]
Tu solución:       0.000609 segundos | Duplicados: [1505, 3742, 8999, 9001, 12000]

Ganancia en eficiencia: 3239x más rápido


---

## Problema 2: Cálculo de Promedios Móviles

Para el análisis técnico, el fondo necesita calcular el **promedio móvil** de los precios de sus activos. Esto suaviza la volatilidad y permite identificar tendencias. El promedio móvil de ventana $k$ para el día $i$ es:

$$\text{PM}_i = \frac{1}{k} \sum_{j=i}^{i+k-1} \text{precio}_j$$

### Código heredado (ineficiente)

Analiza la siguiente función. Identifica:
- ¿Cuál es la complejidad actual en notación Big O? (pista: hay dos variables relevantes)
- ¿Qué operación se está repitiendo innecesariamente?
- ¿A qué complejidad se puede reducir y con qué técnica matemática?

In [5]:
# ============================================================
# CÓDIGO HEREDADO — NO MODIFICAR
# ============================================================

def promedio_movil_ineficiente(datos, k):
    """
    Calcula el promedio móvil recalculando la suma de la ventana cada vez.
    
    Parámetros:
    datos (list): Serie de tiempo de precios.
    k (int): Tamaño de la ventana móvil.
    
    Retorna:
    list: Serie suavizada con los promedios móviles.
    """
    promedios = []
    for i in range(len(datos) - k + 1):
        ventana = datos[i : i + k]
        promedio = sum(ventana) / k
        promedios.append(promedio)
    return promedios

### Tu tarea

Crea una función `promedio_movil_eficiente(datos, k)` que produzca los mismos resultados. Piensa: al deslizar la ventana un día hacia adelante, ¿realmente necesitas sumar *todos* los elementos de nuevo? ¿Qué cambia entre una ventana y la siguiente?

In [6]:
# ============================================================
# TU SOLUCIÓN
# ============================================================

def promedio_movil_eficiente(datos, k):
    """
    Calcula el promedio móvil usando el patrón 'Running Sum' (Suma Continua).
    Al avanzar un día, suma el nuevo precio y resta el que sale de la ventana.
    
    Parámetros:
    datos (list): Serie de tiempo de precios.
    k (int): Tamaño de la ventana móvil.
    
    Retorna:
    list: Serie suavizada con los promedios móviles.
    """
    if not datos or k <= 0 or k > len(datos):
        return []
    
    promedios = []
    # Calculamos la suma inicial UNA sola vez
    suma_actual = sum(datos[:k])
    promedios.append(suma_actual / k)
    
    # Iteramos sobre el resto: sumamos el que entra, restamos el que sale
    for i in range(k, len(datos)):
        suma_actual = suma_actual + datos[i] - datos[i - k]
        promedios.append(suma_actual / k)
    
    return promedios

### Solución — Análisis de Complejidad

| | Código Heredado | Código Optimizado |
| :--- | :--- | :--- |
| **Complejidad** | $O(n \times k)$ | $O(n)$ |
| **Razón** | Para cada una de las $n - k + 1$ posiciones, `sum(ventana)` recorre $k$ elementos. Con $n = 20{,}000$ y $k = 1{,}000$, son ~20 millones de sumas. | Calcula `sum()` una sola vez para la primera ventana. Luego, cada paso siguiente es $O(1)$: suma el nuevo elemento y resta el que sale. |
| **Estrategia** | Recálculo completo en cada iteración | **Running Sum** — Actualización incremental aprovechando que ventanas consecutivas comparten $k - 1$ elementos. |

In [7]:
# ============================================================
# COMPARACIÓN DE RESULTADOS
# ============================================================

print("--- Problema 2: Promedios Móviles ---\n")

t0 = time.time()
res_heredado_pm = promedio_movil_ineficiente(precios, VENTANA)
t1 = time.time()
tiempo_heredado_pm = t1 - t0
print(f"Código heredado:   {tiempo_heredado_pm:.4f} segundos | Promedios calculados: {len(res_heredado_pm):,}")

t0 = time.time()
res_eficiente_pm = promedio_movil_eficiente(precios, VENTANA)
t1 = time.time()
tiempo_eficiente_pm = t1 - t0
print(f"Tu solución:       {tiempo_eficiente_pm:.6f} segundos | Promedios calculados: {len(res_eficiente_pm):,}")

# Verificación (con tolerancia por punto flotante)
assert len(res_heredado_pm) == len(res_eficiente_pm), "ERROR: Diferente cantidad de promedios."
for a, b in zip(res_heredado_pm, res_eficiente_pm):
    assert abs(a - b) < 1e-9, f"ERROR: Discrepancia en valores: {a} vs {b}"

speedup_2 = tiempo_heredado_pm / tiempo_eficiente_pm if tiempo_eficiente_pm > 0 else float('inf')
print(f"\nGanancia en eficiencia: {speedup_2:.0f}x más rápido")

--- Problema 2: Promedios Móviles ---

Código heredado:   0.0628 segundos | Promedios calculados: 19,001
Tu solución:       0.001027 segundos | Promedios calculados: 19,001

Ganancia en eficiencia: 61x más rápido


---

## Problema 3: Reconciliación de Clientes (Compliance)

Por regulación, el fondo debe verificar diariamente si alguno de sus clientes aparece en la lista de personas sancionadas. Esto implica cruzar dos listas grandes: la cartera de clientes y el registro de sancionados.

### Código heredado (ineficiente)

Analiza la siguiente función. Identifica:
- ¿Cuál es la complejidad actual en notación Big O? (¿dónde está el loop oculto?)
- ¿Qué estructura de datos te permitiría eliminar el loop interno?
- ¿A qué complejidad se puede reducir?

In [8]:
# ============================================================
# CÓDIGO HEREDADO — NO MODIFICAR
# ============================================================

def clientes_comunes_ineficiente(lista_a, lista_b):
    """
    Encuentra los clientes que están en ambas listas.
    Usar 'in' sobre una lista obliga a Python a escanearla elemento por elemento.
    
    Parámetros:
    lista_a (list): Clientes del fondo.
    lista_b (list): Clientes sancionados.
    
    Retorna:
    list: IDs de clientes comunes.
    """
    comunes = []
    for cliente in lista_a:
        if cliente in lista_b:    # <-- 'in' sobre una lista es O(m)
            comunes.append(cliente)
    return comunes

### Tu tarea

Crea una función `clientes_comunes_eficiente(lista_a, lista_b)` que devuelva los mismos resultados. Piensa: ¿qué estructura de datos convierte la búsqueda `in` de $O(m)$ a $O(1)$?

In [9]:
# ============================================================
# TU SOLUCIÓN
# ============================================================

def clientes_comunes_eficiente(lista_a, lista_b):
    """
    Encuentra clientes comunes usando un Set para búsqueda instantánea.
    
    Parámetros:
    lista_a (list): Clientes del fondo.
    lista_b (list): Clientes sancionados.
    
    Retorna:
    list: IDs de clientes comunes.
    """
    # Convertimos lista_b a Set: O(m)
    set_b = set(lista_b)
    
    comunes = []
    for cliente in lista_a:
        if cliente in set_b:    # <-- 'in' sobre un set es O(1)
            comunes.append(cliente)
    return comunes

### Solución — Análisis de Complejidad

| | Código Heredado | Código Optimizado |
| :--- | :--- | :--- |
| **Complejidad** | $O(n \times m)$ | $O(n + m)$ |
| **Razón** | Para cada uno de los $n$ clientes de la cartera, el operador `in` recorre los $m$ clientes sancionados secuencialmente (loop oculto). Con $n = m = 15{,}000$, son 225 millones de comparaciones. | Convertir `lista_b` a `set` cuesta $O(m)$ (se hace una vez). Luego, cada búsqueda `in set_b` es $O(1)$ gracias a las tablas hash. Total: $O(m) + O(n) = O(n + m)$. |
| **Estrategia** | Loop oculto con `in` sobre lista | **Estructuras nativas** — Convertir a `set` para búsqueda en $O(1)$. |

In [10]:
# ============================================================
# COMPARACIÓN DE RESULTADOS
# ============================================================

print("--- Problema 3: Reconciliación de Clientes ---\n")

t0 = time.time()
res_heredado_cl = clientes_comunes_ineficiente(clientes_cartera, clientes_sancionados)
t1 = time.time()
tiempo_heredado_cl = t1 - t0
print(f"Código heredado:   {tiempo_heredado_cl:.4f} segundos | Alertas: {len(res_heredado_cl):,}")

t0 = time.time()
res_eficiente_cl = clientes_comunes_eficiente(clientes_cartera, clientes_sancionados)
t1 = time.time()
tiempo_eficiente_cl = t1 - t0
print(f"Tu solución:       {tiempo_eficiente_cl:.6f} segundos | Alertas: {len(res_eficiente_cl):,}")

# Verificación
assert res_heredado_cl == res_eficiente_cl, "ERROR: Los resultados no coinciden."

speedup_3 = tiempo_heredado_cl / tiempo_eficiente_cl if tiempo_eficiente_cl > 0 else float('inf')
print(f"\nGanancia en eficiencia: {speedup_3:.0f}x más rápido")

--- Problema 3: Reconciliación de Clientes ---

Código heredado:   1.2159 segundos | Alertas: 5,940
Tu solución:       0.000720 segundos | Alertas: 5,940

Ganancia en eficiencia: 1689x más rápido


---

## Problema Final: Consolidación — El Reporte Completo

Ahora que optimizaste cada función individualmente, es hora de consolidar todo en una función `generar_reporte_eficiente()` que reemplace al script heredado.

### Código heredado (ineficiente)

Esta función agrega las tres tareas anteriores. Observa cómo llama a las funciones ineficientes:

In [11]:
# ============================================================
# CÓDIGO HEREDADO — NO MODIFICAR
# ============================================================

def generar_reporte_ineficiente(txs, precios_hist, clientes_cart, clientes_sanc, ventana):
    """
    Genera un reporte financiero ejecutando tres procesos con algoritmos de fuerza bruta.
    
    Complejidad Total: O(N^2) + O(P * V) + O(C * S)
    
    Parámetros:
    txs (list): Lista de IDs de transacciones.
    precios_hist (list): Serie de tiempo de precios.
    clientes_cart (list): IDs de clientes del fondo.
    clientes_sanc (list): IDs de clientes sancionados.
    ventana (int): Días para el promedio móvil.
    
    Retorna:
    dict: Resumen del reporte.
    """
    print("  [Ineficiente] 1. Buscando transacciones duplicadas...")
    duplicados = buscar_duplicados_fuerza_bruta(txs)
    
    print("  [Ineficiente] 2. Calculando promedios móviles...")
    promedios_mov = promedio_movil_ineficiente(precios_hist, ventana)
    
    print("  [Ineficiente] 3. Cruzando carteras de clientes...")
    alertas = clientes_comunes_ineficiente(clientes_cart, clientes_sanc)
    
    return {
        "num_duplicados": len(duplicados),
        "total_promedios_calculados": len(promedios_mov),
        "num_alertas_cumplimiento": len(alertas)
    }

### Tu tarea

Crea una función `generar_reporte_eficiente()` que haga exactamente lo mismo pero usando tus funciones optimizadas. Debe recibir los mismos parámetros y devolver un diccionario con la misma estructura.

In [12]:
# ============================================================
# TU SOLUCIÓN
# ============================================================

def generar_reporte_eficiente(txs, precios_hist, clientes_cart, clientes_sanc, ventana):
    """
    Genera el mismo reporte financiero utilizando algoritmos optimizados.
    
    Complejidad Total: O(N) + O(P) + O(C + S)
    
    Parámetros:
    txs (list): Lista de IDs de transacciones.
    precios_hist (list): Serie de tiempo de precios.
    clientes_cart (list): IDs de clientes del fondo.
    clientes_sanc (list): IDs de clientes sancionados.
    ventana (int): Días para el promedio móvil.
    
    Retorna:
    dict: Resumen del reporte.
    """
    print("  [Eficiente] 1. Buscando transacciones duplicadas...")
    duplicados = buscar_duplicados_eficiente(txs)
    
    print("  [Eficiente] 2. Calculando promedios móviles...")
    promedios_mov = promedio_movil_eficiente(precios_hist, ventana)
    
    print("  [Eficiente] 3. Cruzando carteras de clientes...")
    alertas = clientes_comunes_eficiente(clientes_cart, clientes_sanc)
    
    return {
        "num_duplicados": len(duplicados),
        "total_promedios_calculados": len(promedios_mov),
        "num_alertas_cumplimiento": len(alertas)
    }

### Solución — Análisis de Complejidad del Reporte Completo

| Tarea | Complejidad Heredada | Complejidad Optimizada | Estrategia |
| :--- | :--- | :--- | :--- |
| Duplicados | $O(n^2)$ | $O(n)$ | Diccionario como registro |
| Promedios Móviles | $O(n \times k)$ | $O(n)$ | Running Sum |
| Cruce de Clientes | $O(n \times m)$ | $O(n + m)$ | Set para búsqueda $O(1)$ |
| **Total** | **$O(n^2) + O(nk) + O(nm)$** | **$O(n) + O(n) + O(n+m)$** | **Combinación de las tres estrategias** |

La complejidad total del reporte pasa de estar dominada por términos cuadráticos a ser completamente **lineal**.

In [13]:
# ============================================================
# COMPARACIÓN FINAL: REPORTE COMPLETO
# ============================================================

print("=" * 60)
print("  AUDITORÍA FINAL: REPORTE HEREDADO vs. REPORTE OPTIMIZADO")
print("=" * 60)

# Reporte ineficiente
print("\n--- Reporte Ineficiente (Código Heredado) ---")
t0 = time.time()
rep_ineficiente = generar_reporte_ineficiente(
    transacciones, precios, clientes_cartera, clientes_sancionados, VENTANA
)
t1 = time.time()
tiempo_inef = t1 - t0
print(f">> Tiempo total: {tiempo_inef:.4f} segundos")

# Reporte eficiente
print("\n--- Reporte Eficiente (Tu Solución) ---")
t0 = time.time()
rep_eficiente = generar_reporte_eficiente(
    transacciones, precios, clientes_cartera, clientes_sancionados, VENTANA
)
t1 = time.time()
tiempo_ef = t1 - t0
print(f">> Tiempo total: {tiempo_ef:.6f} segundos")

# Verificación de integridad
assert rep_ineficiente["num_duplicados"] == rep_eficiente["num_duplicados"], \
    "ERROR: Discrepancia en duplicados"
assert rep_ineficiente["num_alertas_cumplimiento"] == rep_eficiente["num_alertas_cumplimiento"], \
    "ERROR: Discrepancia en alertas de cumplimiento"
assert rep_ineficiente["total_promedios_calculados"] == rep_eficiente["total_promedios_calculados"], \
    "ERROR: Discrepancia en promedios calculados"

# Resumen
speedup_total = tiempo_inef / tiempo_ef if tiempo_ef > 0 else float('inf')
print(f"\n{'=' * 60}")
print(f"  RESULTADOS DE LA AUDITORÍA")
print(f"{'=' * 60}")
print(f"  Verificación de integridad: EXITOSA (mismos resultados)")
print(f"  Duplicados encontrados:     {rep_eficiente['num_duplicados']}")
print(f"  Promedios calculados:       {rep_eficiente['total_promedios_calculados']:,}")
print(f"  Alertas de cumplimiento:    {rep_eficiente['num_alertas_cumplimiento']:,}")
print(f"{'=' * 60}")
print(f"  Tiempo heredado:    {tiempo_inef:.4f} segundos")
print(f"  Tiempo optimizado:  {tiempo_ef:.6f} segundos")
print(f"  Speedup total:      {speedup_total:.0f}x más rápido")
print(f"{'=' * 60}")

  AUDITORÍA FINAL: REPORTE HEREDADO vs. REPORTE OPTIMIZADO

--- Reporte Ineficiente (Código Heredado) ---
  [Ineficiente] 1. Buscando transacciones duplicadas...
  [Ineficiente] 2. Calculando promedios móviles...
  [Ineficiente] 3. Cruzando carteras de clientes...
>> Tiempo total: 3.2400 segundos

--- Reporte Eficiente (Tu Solución) ---
  [Eficiente] 1. Buscando transacciones duplicadas...
  [Eficiente] 2. Calculando promedios móviles...
  [Eficiente] 3. Cruzando carteras de clientes...
>> Tiempo total: 0.002187 segundos

  RESULTADOS DE LA AUDITORÍA
  Verificación de integridad: EXITOSA (mismos resultados)
  Duplicados encontrados:     5
  Promedios calculados:       19,001
  Alertas de cumplimiento:    5,940
  Tiempo heredado:    3.2400 segundos
  Tiempo optimizado:  0.002187 segundos
  Speedup total:      1481x más rápido


---

## Resumen de Ganancias por Problema

Ejecuta la siguiente celda para ver un resumen consolidado de las ganancias en eficiencia de cada problema.

In [14]:
print("\n" + "=" * 60)
print("  TABLA RESUMEN DE GANANCIAS EN EFICIENCIA")
print("=" * 60)
print(f"{'Problema':<30} {'Heredado':>12} {'Optimizado':>12} {'Speedup':>10}")
print("-" * 64)
print(f"{'1. Duplicados':<30} {tiempo_heredado:>11.4f}s {tiempo_eficiente:>11.6f}s {speedup_1:>9.0f}x")
print(f"{'2. Promedios Móviles':<30} {tiempo_heredado_pm:>11.4f}s {tiempo_eficiente_pm:>11.6f}s {speedup_2:>9.0f}x")
print(f"{'3. Cruce de Clientes':<30} {tiempo_heredado_cl:>11.4f}s {tiempo_eficiente_cl:>11.6f}s {speedup_3:>9.0f}x")
print("-" * 64)
print(f"{'TOTAL (Reporte Completo)':<30} {tiempo_inef:>11.4f}s {tiempo_ef:>11.6f}s {speedup_total:>9.0f}x")
print("=" * 60)


  TABLA RESUMEN DE GANANCIAS EN EFICIENCIA
Problema                           Heredado   Optimizado    Speedup
----------------------------------------------------------------
1. Duplicados                       1.9732s    0.000609s      3239x
2. Promedios Móviles                0.0628s    0.001027s        61x
3. Cruce de Clientes                1.2159s    0.000720s      1689x
----------------------------------------------------------------
TOTAL (Reporte Completo)            3.2400s    0.002187s      1481x
